# Global Information 🌍

In this notebook data analysis study was performed on a given data set which consists of several information about countries of the world.

The breakdown of the study is as below

<ul>
<li>Data exploration</li>
<li>Data transformation</li>
<li>Interpretation</li>
<li>Clustering</li>
</ul>

Data Source: https://www.kaggle.com/datasets/nelgiriyewithana/countries-of-the-world-2023

In [ ]:
import warnings
import numpy as numpy
import pandas as pandas
import matplotlib.pyplot as pyplot
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
import plotly.express as px

In [ ]:
warnings.filterwarnings('ignore')
pandas.set_option('display.max_columns', None)
pandas.set_option('display.max_rows', None)

# Data Exploration

In [ ]:
RawData = pandas.read_csv('/kaggle/input/countries-of-the-world-2023/world-data-2023.csv')
RawData.sample(10)

In [ ]:
RawData.info()

# Data Transformation

The columns that contain numerical values are defined as objects rather than float or integer data types. Also there are many non-numeric characters next to the values hence to make calculations is inappropriate with this data set. Firstly the data type of columns considered as numerical variables are changed. Then the non-numeric characters are removed from value field.

In [ ]:
DataSet = RawData

NumericalVariables = ['Density\n(P/Km2)', 'Agricultural Land( %)', 'Land Area(Km2)', 'Armed Forces size', 'Birth Rate', 'Co2-Emissions', 'CPI', 'CPI Change (%)', 'Fertility Rate', 'Forested Area (%)', 'Gasoline Price', 'GDP', 'Gross primary education enrollment (%)', 'Gross tertiary education enrollment (%)', 'Infant mortality', 'Life expectancy', 'Maternal mortality ratio', 'Minimum wage', 'Out of pocket health expenditure', 'Physicians per thousand', 'Population', 'Population: Labor force participation (%)', 'Tax revenue (%)', 'Total tax rate', 'Unemployment rate', 'Urban_population']

for variable in NumericalVariables:
    if (DataSet[variable].dtypes == 'object'):
        DataSet[variable] = DataSet[variable].str.replace(',', '')
        DataSet[variable] = DataSet[variable].str.replace('%', '')
        DataSet[variable] = DataSet[variable].str.replace('$', '')
        DataSet[variable] = DataSet[variable].astype(float)

for variable in NumericalVariables:
    DataSet[variable].fillna(DataSet[variable].mean(), inplace=True)

DataSet.sample(10)

In [ ]:
DataSet.info()

# Correlation Matrix

In [ ]:
CorrelationMatrix = DataSet[NumericalVariables].corr()

pyplot.figure(figsize=(15,10))

mask = numpy.triu(numpy.ones_like(CorrelationMatrix, dtype=bool))

sns.heatmap(CorrelationMatrix,
            cmap='RdBu_r',
            annot=True,
            fmt='.2f',
            vmin=-1, vmax=1)

pyplot.show()

Regarding to common opinion, correlation does not mean causation. However this is not an obstacle for making some interpretations about the correlation matrix.

<ul>
<li>GDP vs CO2 Emissions</li>
<p>GDP (Gross Domestic Product) is the total monetary or market value of all the finished goods and services produced of a country. GDP is a very common indicator of a country's economic situation. There is a strong positive relation between CO2 Emissions and GDP can tell us maybe the world's biggest economies are responsible for global warming.</p>

<li>Gross Tertiary Education Enrollment vs Birth Rate</li>
<p>Tertiary education is education activities for people above school age such as university, college. Correlation value shows us there is a negative relationship between theese two variables. If education levels up, then birth rate goes down. It seems like people who have higher education level does not prefer to reproduce.</p>

<li>Life Expectancy vs Birth Rate</li>
<p>Another comparison involving birth rate is against to life expectancy. Countries with high birth rate have lower life expectancy than other countries. This case can be quite explainable when we consider there is a trouble in allocation of scarce medical resources in populated countries.</p>

</ul>

# Data Preprocessing

Each numerical variable will be a axis of clustering space. Since distance calculation is a crucial function in K Means Clustering Algorithm, we have to scale the values of variables.

In [ ]:
ClusteringVariables = ['CPI', 'Life expectancy', 'Gross tertiary education enrollment (%)']

GeographicIndicators = ['Density\n(P/Km2)', 'Agricultural Land( %)', 'Forested Area (%)']

EconomicIndicators = ['GDP', 'CPI', 'Unemployment rate']


Axes = DataSet[NumericalVariables]
Scaler = MinMaxScaler()
Axes = Scaler.fit_transform(Axes)
Axes = pandas.DataFrame(Axes, columns=[NumericalVariables])
Axes.sample(10)

# The Elbow Method

K Means Clustering Algorithm is a simple and flexible method. However there a challenge to determine parameter "K" which is the number of clusters in clustering space. The Elbow Method is a convenient method to select the value of "K".

In [ ]:
InertiaList = []
NumberOfClustersRange = range(1, 10)

for i in range(1, 10):
    clusteringModel = KMeans(n_clusters=i)
    clusteringModel.fit(Axes)
    InertiaList.append(clusteringModel.inertia_)

pyplot.plot(range(1, 10), InertiaList)
pyplot.xlabel('Number Of Clusters')
pyplot.ylabel('Inertia')
pyplot.title('The Elbow Method')
pyplot.show()

# Clustering

In [ ]:
KMeansClustering = KMeans(n_clusters=4)

KMeansClustering.fit(Axes)

ClusterAssignments = pandas.DataFrame({'Country': DataSet.Country, 'Cluster': KMeansClustering.labels_})

fig = px.choropleth(ClusterAssignments,
                    locations='Country',
                    locationmode='country names',
                    color='Cluster',
                    hover_name='Country',                    
                    title = 'Clusters',
                    color_continuous_scale='YlGnBu',
                    width=1000,
                    height=600
                    )
fig.show()

#ClusterAssignments.sort_values(by=['Cluster'], ascending=False)